# DEBUG: StandUp4AI Feature Extraction

Step-by-step debugging.

In [ ]:
# 1. Mount Drive and find folder
from google.colab import drive
drive.mount('/content/drive')

import os

# Try to find standup4ai folder
paths_to_try = [
    '/content/drive/MyDrive/standup4ai',
    '/content/drive/Shareddrives/standup4ai',
]

BASE = None
for p in paths_to_try:
    if os.path.exists(p):
        BASE = p
        print(f'Found at: {p}')
        break

if BASE is None:
    # List what's in My Drive
    mydrive = '/content/drive/MyDrive'
    if os.path.exists(mydrive):
        items = os.listdir(mydrive)
        print(f'My Drive contents ({len(items)} items):')
        for item in sorted(items):
            if 'stand' in item.lower() or 'laugh' in item.lower() or 's4ai' in item.lower():
                print(f'  ** {item}')
            else:
                print(f'    {item}')
else:
    print(f'BASE = {BASE}')
    AUDIO_DIR = os.path.join(BASE, 'audio')
    LABELS_DIR = os.path.join(BASE, 'labels')
    print(f'Audio dir: {AUDIO_DIR}')
    print(f'  exists={os.path.exists(AUDIO_DIR)}')
    print(f'Labels dir: {LABELS_DIR}')
    print(f'  exists={os.path.exists(LABELS_DIR)}')

In [ ]:
# 2. List files and find overlap
if BASE:
    audio_files = [f for f in os.listdir(AUDIO_DIR) if f.endswith('.m4a') or f.endswith('.mp3')]
    label_files = [f for f in os.listdir(LABELS_DIR) if f.endswith('.csv')]
    print(f'Audio files: {len(audio_files)}')
    print(f'Label files: {len(label_files)}')
    
    # Strip extensions to get video IDs
    audio_vids = {f.replace('.m4a','').replace('.mp3','') for f in audio_files}
    label_vids = {f.replace('.csv','') for f in label_files}
    overlap = sorted(audio_vids & label_vids)
    print(f'Overlap (have both): {len(overlap)}')
    print(f'Sample: {overlap[:5]}')


In [ ]:
# 3. Test audio loading with librosa
import subprocess
subprocess.run(['pip', 'install', '-q', 'librosa'], check=True, timeout=60)
import librosa
import numpy as np

if BASE and overlap:
    test_vid = overlap[0]
    audio_path = os.path.join(AUDIO_DIR, f'{test_vid}.m4a')
    print(f'Testing: {test_vid}')
    print(f'Path: {audio_path}')
    print(f'File size: {os.path.getsize(audio_path):,} bytes')
    
    # Try loading first 10 seconds
    try:
        y, sr = librosa.load(audio_path, sr=22050, duration=10.0)
        print(f'Load OK: y.shape={y.shape}, sr={sr}')
        print(f'Duration: {len(y)/sr:.1f}s')
    except Exception as e:
        print(f'Load ERROR: {e}')
        y = None
    
    if y is not None:
        # Try with audioread fallback
        try:
            y2, sr2 = librosa.load(audio_path, sr=22050, duration=3.0, res_type='audioread')
            print(f'audioread OK: y.shape={y2.shape}')
        except Exception as e:
            print(f'audioread ERROR: {e}')

In [ ]:
# 4. Test label CSV loading
import pandas as pd

if BASE and overlap:
    test_vid = overlap[0]
    label_path = os.path.join(LABELS_DIR, f'{test_vid}.csv')
    print(f'Testing: {test_vid}')
    print(f'Path: {label_path}')
    print(f'File size: {os.path.getsize(label_path):,} bytes')
    
    try:
        df = pd.read_csv(label_path)
        print(f'Loaded! shape={df.shape}')
        print(f'Columns: {list(df.columns)}')
        print(f'First 3 rows:')
        print(df.head(3).to_string())
        print(f'Label dist: {df["label"].value_counts().to_dict()}')
    except Exception as e:
        print(f'ERROR: {e}')
        # Read raw
        with open(label_path, 'r') as f:
            lines = f.readlines()[:5]
        print(f'Raw lines: {lines}')

In [ ]:
# 5. Test feature extraction on ONE segment
if BASE and overlap:
    test_vid = overlap[0]
    audio_path = os.path.join(AUDIO_DIR, f'{test_vid}.m4a')
    label_path = os.path.join(LABELS_DIR, f'{test_vid}.csv')
    
    df = pd.read_csv(label_path)
    row = df.iloc[0]
    t0, t1 = float(row['t0']), float(row['t1'])
    label = row['label'].strip()
    print(f'Segment: {t0:.2f}s - {t1:.2f}s, label={label}')
    
    dur = min(t1 - t0, 10.0)
    print(f'Extract duration: {dur:.2f}s')
    
    try:
        y, sr = librosa.load(audio_path, sr=22050, offset=t0, duration=dur, mono=True)
        print(f'Audio loaded: {y.shape}, sr={sr}')
    except Exception as e:
        print(f'Audio load ERROR: {e}')
        y = None
    
    if y is not None and len(y) > 0:
        # F0
        try:
            f0, voiced_flag, _ = librosa.pyin(y, fmin=80, fmax=500, sr=sr, hop_length=512)
            f0 = np.nan_to_num(f0, nan=0)
            print(f'F0 mean={np.mean(f0):.1f}Hz, voiced_rate={np.mean(voiced_flag):.2f}')
        except Exception as e:
            print(f'F0 ERROR: {e}')
        
        # RMS energy
        rms = librosa.feature.rms(y=y, hop_length=512)[0]
        print(f'RMS: mean={np.mean(rms):.4f}, max={np.max(rms):.4f}')
        
        # ZCR
        zcr = librosa.feature.zero_crossing_rate(y, hop_length=512)[0]
        print(f'ZCR: mean={np.mean(zcr):.4f}')
        
        print('SUCCESS!')
    else:
        print('Audio empty or load failed!')

In [ ]:
# 6. Full extraction
if BASE and overlap:
    print(f'Extracting from {len(overlap)} videos...')
    
    X_all, y_all, vids_all = [], [], []
    
    for i, vid in enumerate(overlap):
        audio_path = os.path.join(AUDIO_DIR, f'{vid}.m4a')
        label_path = os.path.join(LABELS_DIR, f'{vid}.csv')
        
        if not os.path.exists(audio_path) or not os.path.exists(label_path):
            continue
        
        try:
            df = pd.read_csv(label_path)
        except:
            continue
        
        for _, seg in df.iterrows():
            try:
                t0, t1 = float(seg['t0']), float(seg['t1'])
                dur = min(t1 - t0, 10.0)
                if dur < 0.1:
                    continue
                
                y, sr = librosa.load(audio_path, sr=22050, offset=t0, duration=dur, mono=True)
                if len(y) < sr * 0.1:
                    continue
                
                f0, voiced_flag, _ = librosa.pyin(y, fmin=80, fmax=500, sr=sr, hop_length=512)
                f0 = np.nan_to_num(f0, nan=0)
                
                rms = librosa.feature.rms(y=y, hop_length=512)[0]
                zcr = librosa.feature.zero_crossing_rate(y, hop_length=512)[0]
                
                feat = np.array([
                    np.mean(f0), np.std(f0), np.mean(voiced_flag),
                    np.mean(rms), np.std(rms), np.max(rms),
                    np.mean(zcr), np.std(zcr),
                    len(y)/sr
                ], dtype=np.float32)
                
                X_all.append(feat)
                y_all.append(1 if seg['label'].strip() == 'risa' else 0)
                vids_all.append(vid)
                
            except Exception as e:
                pass  # Skip bad segments silently
        
        if (i + 1) % 5 == 0:
            print(f'  {i+1}/{len(overlap)} videos, {len(X_all)} samples...')
    
    print(f'\n=== RESULTS ===')
    print(f'Total samples: {len(X_all)}')
    if y_all:
        pos_rate = sum(y_all)/len(y_all)
        print(f'Positive: {sum(y_all)} ({pos_rate:.1%})')
    else:
        print('Positive: 0 (0%)')
    print(f'Videos: {len(set(vids_all))}')
    
    if X_all:
        X = np.array(X_all)
        y = np.array(y_all)
        print(f'Features: {X.shape[1]}-dim')
        print('READY FOR TRAINING!')
    else:
        print('ERROR: No samples extracted!')

In [ ]:
# 7. If we have data, run training
if 'X' in dir() and len(X_all) > 0:
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import GradientBoostingClassifier
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import f1_score, precision_score, recall_score
    from sklearn.model_selection import GroupKFold
    import json
    
    print('\n=== VIDEO-LEVEL CV ===')
    
    models = {
        'LogReg': LogisticRegression(max_iter=1000, class_weight='balanced', C=0.1),
        'XGBoost': GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42),
    }
    
    n_splits = min(5, len(set(vids_all)))
    gkf = GroupKFold(n_splits=n_splits)
    results = {}
    
    for name, model in models.items():
        f1s, precs, recs = [], [], []
        for tr, te in gkf.split(X, y, vids_all):
            if len(set(y[te])) < 2:
                continue
            sc = StandardScaler()
            Xtr = sc.fit_transform(X[tr])
            Xte = sc.transform(X[te])
            model.fit(Xtr, y[tr])
            pred = model.predict(Xte)
            f1s.append(f1_score(y[te], pred, zero_division=0))
            precs.append(precision_score(y[te], pred, zero_division=0))
            recs.append(recall_score(y[te], pred, zero_division=0))
        
        mean_f1 = np.mean(f1s) if f1s else 0
        std_f1 = np.std(f1s) if f1s else 0
        results[name] = mean_f1
        print(f'{name:12s} F1={mean_f1:.4f} ± {std_f1:.4f}')
    
    print(f'\n{"="*50}')
    print(f'StandUp4AI baseline: F1=0.51')
    for name, f1 in sorted(results.items(), key=lambda x: -x[1]):
        beat = '🏆 BEATS BASELINE' if f1 > 0.51 else ''
        print(f'{name:12s} F1={f1:.4f} {beat}')
    print(f'{"="*50}')
    
    # Save
    result_path = os.path.join(BASE, 'standup4ai_results.json')
    with open(result_path, 'w') as f:
        json.dump({'n_samples': len(X_all), 'n_videos': len(set(vids_all)),
                   'models': results, 'baseline': 0.51}, f, indent=2)
    print(f'\nSaved to {result_path}')
else:
    print('No data - cannot train. Check errors above.')